In [1]:
import sys 
import os
from pathlib import Path
import pandas as pd

#from word_embeddings import *
import torch

In [2]:
from csv import QUOTE_NONE
import sys
import csv

maxInt = sys.maxsize

while True:
    # decrease the maxInt value by factor 10 
    # as long as the OverflowError occurs.

    try:
        csv.field_size_limit(maxInt)
        break
    except OverflowError:
        maxInt = int(maxInt/10)


base_path = Path(os.path.abspath("")).parents[1] / "dataset_creation" / "data"
datasets = {
    "school_shooters": base_path / "school_shooters.csv",
    "manifestos": base_path / "manifestos.csv",
    "stair_twitter_archive": base_path / "stair_twitter_archive.csv",
    "twitter": base_path / "twitter.csv",
    "stream_of_consciousness": base_path / "stream_of_consciousness.csv"
}

schoolshootersinfo_df = pd.read_csv(datasets["school_shooters"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
#manifesto_df = pd.read_csv(datasets["manifestos"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stair_twitter_archive_df = pd.read_csv(datasets["stair_twitter_archive"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
twitter_df = pd.read_csv(datasets["twitter"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stream_of_consciousness_df = pd.read_csv(datasets["stream_of_consciousness"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)

In [3]:
schoolshootersinfo_df["label"] = 1
#manifesto_df["shooter"] = 1
stair_twitter_archive_df["label"] = 1
twitter_df["label"] = 0
stream_of_consciousness_df["label"] = 0

In [4]:
from word_embeddings import preprocess_text, get_glove_word_vectors

ModuleNotFoundError: No module named 'torchtext'

In [ ]:
shooter_df = pd.concat([schoolshootersinfo_df, stair_twitter_archive_df], ignore_index=True)[:100]
non_shooter_df = pd.concat([twitter_df[:100], stream_of_consciousness_df[:100]])
whole_corpus_df = pd.concat([shooter_df, non_shooter_df], ignore_index=True).sample(frac=1)

In [ ]:
whole_corpus_df["text"] = whole_corpus_df["text"].map(lambda a: get_glove_word_vectors(preprocess_text(a)))

In [ ]:
from sklearn.model_selection import train_test_split 

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(whole_corpus_df["text"], whole_corpus_df["label"], test_size=0.1)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1)

In [ ]:
# Check sizes of tensors to shape input to mlp model
shapes = [list(t.shape) for t in whole_corpus_df["text"]]
dic_size = [dims[0] for dims in shapes]
print(shapes)


max_dic_size = max(dic_size)
min_dic_size = min(dic_size)
print(min_dic_size)
print(max_dic_size)
for x in x_train:
    if x.shape[0] == min_dic_size:
        print("x_shape: ", x.shape)
        i = 0
        for _ in x:
            print(_)
            i += 1
            print("i: ", i)

[[3, 50], [367, 50], [7, 50], [25, 50], [114, 50], [11, 50], [53, 50], [260, 50], [19, 50], [359, 50], [13, 50], [33, 50], [3, 50], [6, 50], [387, 50], [247, 50], [41, 50], [269, 50], [302, 50], [271, 50], [13, 50], [308, 50], [37, 50], [210, 50], [41, 50], [14, 50], [5, 50], [15, 50], [10, 50], [34, 50], [321, 50], [284, 50], [13, 50], [333, 50], [15, 50], [332, 50], [14, 50], [2, 50], [11, 50], [9, 50], [29, 50], [90, 50], [188, 50], [3, 50], [175, 50], [13, 50], [12, 50], [14, 50], [9, 50], [275, 50], [239, 50], [343, 50], [18, 50], [585, 50], [11, 50], [5, 50], [17, 50], [154, 50], [13, 50], [6, 50], [245, 50], [11, 50], [279, 50], [9, 50], [14, 50], [321, 50], [9, 50], [7, 50], [211, 50], [5, 50], [4, 50], [6, 50], [266, 50], [428, 50], [10, 50], [13, 50], [261, 50], [339, 50], [14, 50], [314, 50], [327, 50], [11, 50], [10, 50], [389, 50], [8, 50], [11, 50], [7, 50], [10, 50], [10, 50], [42, 50], [8, 50], [243, 50], [4, 50], [473, 50], [10, 50], [79, 50], [10, 50], [7, 50], [18, 5

In [ ]:
# design model

import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"


emb_dim = 50
out_dim = 1

model = nn.Sequential(
    nn.Linear(emb_dim, 100, device=device),
    nn.ReLU(),
    nn.Linear(100, 50, device=device),
    nn.ReLU(),
    nn.Linear(50, out_dim, device=device),
    nn.Softmax()
)

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Training
epochs = 20

for n in range(num_epochs):
    y_pred = model(X)
    loss = loss_fn(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()